# 제품 유형 판단 실험 (image-typing)

`image-typing`은 제품 `Typer.TypeImage`(apps/api/internal/processing/typing.go)를 가리킨다. 답은 `text` · `receipt` · `unsupported` · `ambiguous` 넷이다.

- 분류(`Classifier.Classify`의 category · suggestedAction)와 **다른 계약**이다. 분류 결과를 유형 정답이나 예측으로 바꾸지 않는다
- Go에 이 과제가 아직 없다. 그래서 replay 기록 없이 연구 기록(`<runId>.research.jsonl` · `.run.json`)만 남긴다. 계약 밖의 답(예: `document`)도 원래 값을 잃지 않고 contract-error로 남는다
- fixture는 `fixtures/sample-image-typing`(lab 안, `pnpm eval list`가 읽지 않음)

기준선: 제품 유형 판단 지시문(`prompts/product-typing.md`) · 운영 모델 — 공급자 호출 전까지 **미측정**. 규칙 바닥값 `always-text`는 모델 성능이 아니다.

`MODE`와 외부 전송 조건은 `text/01_ocr_translation_lab.ipynb`와 같다.

In [ ]:
from pathlib import Path

import pandas as pd

from snapdone_eval_lab import tasks as t
from snapdone_eval_lab.datasets import load_research_fixture, repository_root
from snapdone_eval_lab.experiments import load_prompt
from snapdone_eval_lab.research import RunPlan, records_frame, transfer_blockers, unique_run_id

ROOT = repository_root()
LAB = ROOT / "tools/evals/lab"
OUT = LAB / "out"

# ---- 실행 설정 -------------------------------------------------------------
MODE = "fixture"                  # "fixture" · "dry-run" · "provider"
ALLOW_EXTERNAL = False
PROVIDER = "anthropic"
MODEL = "claude-haiku-5-5"
BASE_URL = None
MAX_CALLS = 10
SPLIT = "dev"

PLAN = RunPlan(
    hypothesis="제품 유형 판단 지시문이 네 유형의 경계(결제 완료 · 결제 불명 · 글자 없음)를 가른다",
    changed_variable="없음(기준선)",
    controls=["같은 case 선택", "같은 응답 schema"],
    comparison=["규칙 바닥값: always-text"],
    selection_reason="dev 전체(작은 합성 fixture)",
)
if MODE == "provider" and not ALLOW_EXTERNAL:
    raise PermissionError("provider 모드는 ALLOW_EXTERNAL = True(외부 전송 승인)가 필요하다")

In [ ]:
typing = load_research_fixture("sample-image-typing", SPLIT)
CASES = sorted(typing.frame["id"])
if MODE == "provider" and (blocked := transfer_blockers(typing, CASES)):
    raise PermissionError(f"외부 전송 태그가 없는 case: {blocked}")
typing.frame[["id", "image_type", "ambiguity", "source_group", "tags"]]

In [ ]:
inputs = t.image_inputs(typing, CASES)
if MODE == "provider":
    candidate = t.Candidate("lab-typing-product", "fixed-prompt-model", PROVIDER, MODEL, load_prompt(LAB / "prompts/product-typing.md"), base_url=BASE_URL, prompt_ref="tools/evals/lab/prompts/product-typing.md")
    how = t.provider_caller(candidate, t.IMAGE_TYPING)
else:
    candidate, how = t.Candidate("smoke-image-typing", "smoke-fixture"), t.fixture_caller(LAB / "fixtures/smoke/image-typing.json")
runs = {
    "candidate": (candidate, t.run_task(t.IMAGE_TYPING, how, inputs, run_id=unique_run_id("lab-typing", OUT), max_calls=MAX_CALLS, dry_run=MODE == "dry-run")),
    "floor": (t.Candidate("rule-always-text", "rule-baseline"), t.run_task(t.IMAGE_TYPING, t.constant_type_baseline("text"), inputs, run_id=unique_run_id("lab-typing-floor", OUT))),
}

In [ ]:
# 정답 유형과 답한 유형을 나란히 센다(연구 진단). 계약 밖 답도 원래 값으로 보인다. 공식 점수가 아니다.
for name, (_, records) in runs.items():
    frame = records_frame(records)
    frame["answer"] = frame["output"].map(lambda o: o.get("imageType") if isinstance(o, dict) else None)
    frame = frame.merge(typing.frame[["id", "image_type"]], left_on="case_id", right_on="id")
    print(name)
    display(pd.crosstab(frame["image_type"], frame["answer"].fillna("(" + frame["state"] + ")"), rownames=["gold"], colnames=["answer"]))
    display(frame.loc[frame["state"] != "answered", ["case_id", "state", "problem", "answer"]])

In [ ]:
# 연구 기록만 쓴다. Go 과제가 생기면 이 파일이 replay 기록으로 옮겨 갈 입력이다.
for name, (candidate, records) in runs.items():
    paths = t.export_run(records, inputs, spec=t.IMAGE_TYPING, candidate=candidate, plan=PLAN, dataset=typing, out=OUT, max_calls=MAX_CALLS, dry_run=MODE == "dry-run")
    print(name, {k: (v.name if v else None) for k, v in paths.items()})